# Plot a saved simulation — no fitting

Choose one completed results subfolder below. Load only locally generated, trusted archives. All coefficient, prediction, CV, and optimizer-history plots use the saved arrays. Connectivity is loaded only if explicitly requested.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_DIR = Path.cwd() / "model"
if not (MODEL_DIR / "simulation_results.py").exists():
    MODEL_DIR = Path.cwd()
sys.path.insert(0, str(MODEL_DIR))
from simulation_results import (
    RESULTS_DIR, betas_from_blocks, run_simulation_experiment,
    load_simulation_experiment, plot_saved_betas,
)
from config import PARAMS

from plot_utils import (
    plot_lambda_heatmap, plot_fold_r2_by_lambda_pair,
    plot_train_test_grid_all_lambdas, plot_X1_X2_heatmaps,
)


In [ ]:
completed_runs = []
for path in sorted(RESULTS_DIR.glob("*/metadata.json")):
    metadata = json.loads(path.read_text())
    if metadata.get("schema_version") == 1 and metadata.get("status") == "complete":
        completed_runs.append((metadata["created_utc"], path.parent))
completed_runs.sort()
for index, (_, path) in enumerate(completed_runs):
    print(index, path.name)
if not completed_runs:
    raise RuntimeError("No completed saved experiments yet. Run simulation_experiments.ipynb first.")

RUN_INDEX = -1  # Latest completed run; change to an index printed above.
RUN_DIR = completed_runs[RUN_INDEX][1]
saved = load_simulation_experiment(RUN_DIR)
print("Loaded:", RUN_DIR)
display(saved["cv_summary"])

In [ ]:
plot_saved_betas(saved, RUN_DIR / "figures" / "true_vs_estimated_betas.png")
plt.show()

In [ ]:
plot_lambda_heatmap(saved["cv_summary"], metric="Test R2")
plot_fold_r2_by_lambda_pair(saved["cv_results"])
# Optional detailed grid (can be large for many lambda pairs):
# plot_train_test_grid_all_lambdas(saved["cv_results"])

In [ ]:
model = saved["model"]
y_true, y_pred = model["y_true"], model["y_pred"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_true, y_pred, s=12, alpha=0.6)
lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], "k--")
axes[0].set(xlabel="Simulated outcome", ylabel="Fitted outcome", title="Full-data fit (in-sample)")
axes[1].scatter(y_pred, y_true-y_pred, s=12, alpha=0.6)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="Fitted outcome", ylabel="Residual")
plt.tight_layout()
plt.show()

In [ ]:
history = saved["history"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for candidate, losses in history["loss_history"].items():
    axes[0].plot(losses, label=f"Start {candidate+1}")
    axes[1].semilogy(np.maximum(history["stationarity_history"][candidate], 1e-16),
                     label=f"Start {candidate+1}")
axes[0].set_ylabel("Original penalized objective")
axes[1].set_ylabel("Stationarity")
axes[1].axhline(saved["metadata"]["parameters"]["TOL"], color="black", linestyle="--")
for ax in axes:
    ax.set_xlabel("Iteration")
    ax.legend()
plt.tight_layout()
plt.show()

trajectory = history["trajectory"][history["selected_candidate"]]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for block, ax in enumerate(axes, 1):
    ax.plot(trajectory["iterations"], trajectory[f"beta{block}"][:, :, 0])
    ax.set(xlabel="Iteration", ylabel="Coefficient", title=f"Beta {block} trajectory")
plt.tight_layout()
plt.show()

# Optional connectivity plots, if saved:
# with_connectivity = load_simulation_experiment(RUN_DIR, load_connectivity=True)
# plot_X1_X2_heatmaps(with_connectivity["simulation"]["X1"],
#                    with_connectivity["simulation"]["X2"], subject=0)